In [ ]:

import win32com.client
import csv

NETWORK = r"D:\BTP_Vissim\simulation.inpx"

APPROACH_LANES = {"1-1", "1-2"}

SIMULATION_STEPS = 3900

vissim = win32com.client.DispatchEx("Vissim.Vissim")
vissim.LoadNet(NETWORK)

vissim.Graphics.CurrentNetworkWindow.SetAttValue("QuickMode", 1)

seen_vehicles = set()
vehicle_count = 0

dimensions = {}

vehicle_data = []

for step in range(SIMULATION_STEPS):

    vissim.Simulation.RunSingleStep()

    sim_time = step + 1
    vehicles = vissim.Net.Vehicles.GetAll()

    for i in range(len(vehicles)):

        vehicle = vehicles[i]

        vehicle_id = int(vehicle.AttValue("No"))
        lane_id = str(vehicle.AttValue("Lane")).strip()

        if lane_id not in APPROACH_LANES:
            continue
        if vehicle_id in seen_vehicles:
            continue

        seen_vehicles.add(vehicle_id)
        vehicle_count += 1

        vehicle_type_id = int(vehicle.AttValue("VehType"))

        if vehicle_type_id not in dimensions:

            vehicle_type = vissim.Net.VehicleTypes.ItemByKey(
                vehicle_type_id
            )

            dimensions[vehicle_type_id] = {
                "length" : float(vehicle.AttValue("Length")),
                "width" : float(vehicle.AttValue("Width"))
            }

        dims = dimensions[vehicle_type_id]
        record = {
            "time": sim_time,
            "vehicle_id": vehicle_id,
            "lane": lane_id,
            "vehicle_type_id": vehicle_type_id,
            "length_m": dims["length"],
            "width_m": dims["width"],
            "total_vehicles_seen": vehicle_count
        }

        vehicle_data.append(record)

        print(record)

print("\nTotal vehicles detected:", vehicle_count)


In [ ]:

with open("VISSIM_Vehicle_Data.csv", "w", newline="") as file:

    writer = csv.DictWriter(
        file,
        fieldnames=[
            "time",
            "vehicle_id",
            "lane",
            "vehicle_type_id",
            "length_m",
            "width_m",
            "total_vehicles_seen"
        ]
    )

    writer.writeheader()
    writer.writerows(vehicle_data)

print("\nData successfully saved to VISSIM_Vehicle_Data.csv")